# Ultralytics

[Ultralytics](https://docs.ultralytics.com/) is a company and open-source community known for developing state-of-the-art machine learning models, particularly for object detection tasks. They are most famous for the YOLO (You Only Look Once) series of models, which are highly efficient and widely used in real-time object detection. Ultralytics focuses on making cutting-edge deep learning technology accessible through easy-to-use Python libraries, tools, and pre-trained models. Their YOLO models, are popular for applications in industries like security, automation, robotics, and can be used for research!

Example of a paper which employed Yolo from ultralytics -->
https://ieeexplore.ieee.org/document/10426304/


## Yolo history, was also made by Ultralytics

## Recap
* YOLO (You Only Look Once), a popular object detection and image segmentation model, was developed by Joseph Redmon and Ali Farhadi at the University of Washington. Launched in 2015, YOLO quickly gained popularity for its high speed and accuracy.

* * YOLOv2, released in 2016, improved the original model by incorporating batch normalization, anchor boxes, and dimension clusters.
YOLOv3, launched in 2018, further enhanced the model's performance using a more efficient backbone network, multiple anchors and spatial pyramid pooling.
* YOLOv4 was released in 2020, introducing innovations like Mosaic data augmentation, a new anchor-free detection head, and a new loss function.
* YOLOv5 further improved the model's performance and added new features such as hyperparameter optimization, integrated experiment tracking and automatic export to popular export formats.
* YOLOv6 was open-sourced by Meituan in 2022 and is in use in many of the company's autonomous delivery robots.
* YOLOv7 added additional tasks such as pose estimation on the COCO keypoints dataset.
* YOLOv8 released in 2023 by Ultralytics. YOLOv8 introduced new features and improvements for enhanced performance, flexibility, and efficiency, supporting a full range of vision AI tasks,
* YOLOv9 introduces innovative methods like Programmable Gradient Information (PGI) and the Generalized Efficient Layer Aggregation Network (GELAN).
* YOLOv10 is created by researchers from Tsinghua University using the Ultralytics Python package. This version provides real-time object detection advancements by introducing an End-to-End head that eliminates Non-Maximum Suppression (NMS) requirements.
* YOLO11 🚀 NEW: Ultralytics' latest YOLO models delivering state-of-the-art (SOTA) performance across multiple tasks, including detection, segmentation, pose estimation, tracking, and classification, leverage capabilities across diverse AI applications and domains.
* YOLO12 (2025) introduces an attention-centric architecture (Area Attention and R-ELAN blocks) while keeping real-time speed. **In this notebook we use YOLO12 (`yolo12n.pt`) for tracking.**

## Coding

### Install

In [ ]:
! pip install -q ultralytics scikit-learn tslearn

import ultralytics
ultralytics.checks()
import copy
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Download the video from the course GitHub repo
import os
import urllib.request

VIDEO_URL = "https://raw.githubusercontent.com/lorenzo-stacchio/Deep-Learning-and-Computer-Vision-for-Business/main/02-Pytorch%20and%20CV/03_tracking/media/mall_crowd.mp4"
path_video = "mall_crowd.mp4"

if not os.path.exists(path_video):
    urllib.request.urlretrieve(VIDEO_URL, path_video)
print(path_video, os.path.exists(path_video))

In [ ]:
# define output video path
path_video_out = path_video.replace(".mp4", "_out.avi")

In [ ]:
# from ultralytics import YOLO

# # Load an official or custom model
# model = YOLO("yolo12n.pt")  # Load an official YOLO12 Detect model
# # model = YOLO("yolo11n-seg.pt")  # official YOLO12 weights are Detect only: use YOLO11 for Segment
# # model = YOLO("yolo11n-pose.pt")  # official YOLO12 weights are Detect only: use YOLO11 for Pose
# # model = YOLO("path/to/best.pt")  # Load a custom trained model

# # Perform tracking with the model
# # results = model.track(source=path_video, save=True)  # Tracking with default tracker
# results = model.track(source=path_video, save=True, tracker="bytetrack.yaml")  # with ByteTrack

### Yolo inference

In [ ]:
## save results in a list of dictionaries!
from ultralytics import YOLO
import cv2
# from google.colab.patches import cv2_imshow
# Load the YOLO12 model
model = YOLO("yolo12n.pt")

frame_results = []

In [ ]:
# Open the video file
cap = cv2.VideoCapture(path_video)
assert cap.isOpened(), f"Cannot open video: {path_video}"

# Retrieve video properties: width, height, and frames per second
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# Initialize video writer to save the output video with the specified properties
out = cv2.VideoWriter(path_video_out, cv2.VideoWriter_fourcc(*"MJPG"), fps, (w, h))

idx = 0
limit = False
# Loop through the video frames
while cap.isOpened():
    # Read a frame from the video
    success, frame = cap.read()

    if success:
        # Run YOLO12 tracking on the frame, persisting tracks between frames
        results = model.track(frame, persist=True)
                      # save=True,
                      # save_txt=True,
                      # save_conf=True)
        frame_results.append(results)
        # break
        # Visualize the results on the frame
        annotated_frame = results[0].plot()
        
        # print(idx, frame_count//2, idx % (frame_count//2)==0)
        # if idx % (frame_count//2)==0:
          # Display the annotated frame
          # cv2.imshow("", mat=annotated_frame) # if you are in local
          #cv2_imshow(annotated_frame) # if you are in google colab
          # Break the loop if 'q' is pressed
          # if cv2.waitKey(1) & 0xFF == ord("q"):
          # break
          # cv2.destroyAllWindows()

        out.write(annotated_frame)
        

        idx +=1 
        if limit and idx > 50:
            break
        # break
        
    else:
        # Break the loop if the end of the video is reached
        break

# Release the video capture object and close the display window
cap.release()

# Release the video writer and capture objects, and close all OpenCV windows
out.release()

In [ ]:
# first frame results
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
np.set_printoptions(threshold=np.inf)

import seaborn as sns
 
rows = []

for idx_frame, res in enumerate(frame_results):
    # print(len(res))
    # print(res)
    for r in res:
        class_names = r.names
        # print(class_names)
        # visualize all the fields with this 
        # print(r.boxes)
        # print(r.boxes.shape) # --> number of found objects (number_object, predicted_info)
        for idx, box in enumerate(r.boxes): # for each found object
            if box.id is None:  # object detected but not (yet) tracked
                continue
            # print(f"-----------DATA for box {idx}-------------")
            # id
            object_id = box.id.item()  # Tracking ID
            object_cls = int(box.cls.item())  # Class ID
            object_cls_name = class_names[object_cls]  # Class name
            object_xyxy = box.xyxy.tolist()  # Bounding box coordinates
            object_xyxyn = box.xyxyn.tolist()  # Normalized bounding box coordinates

            # Append the new row with all extracted information
            new_row = {
                "frame_id": idx_frame,
                "object_id": object_id,
                "object_xyxy": object_xyxy,
                "object_xyxyn": object_xyxyn,
                "cls": object_cls_name  # Store class name or class ID as needed
            }
            
            rows.append(new_row)
        
    #     break
    # break

# Create the dataframe with all the tracked objects
df = pd.DataFrame(rows, columns=["frame_id", "object_id", "object_xyxy", "object_xyxyn", "cls"])

In [ ]:
df

In [ ]:
df["frame_id"].unique()

In [ ]:
df.groupby("object_id").count()

## Analytics

What to do with this data?

Exercise:


1.   Analyze frame results and find which variable could be interest of your analysis 💡;
2.   Think on some kind of useful analysis that we can do 💡;
3: Implement it 💻



Example of interesting analysis:

* People Counting;
* Spatial Areas of Interests;
* People trajectory analysis;

### People Counting (easy)

In [ ]:
df = df[df["cls"]=="person"] # filter only people
people = df["object_id"].unique()
print(len(people))

### Spatial Areas of Interests

In [ ]:
df['object_xyxy']

In [ ]:
# Extract the center of each bounding box 
df['center_x'] = df['object_xyxy'].apply(lambda xyxy: (xyxy[0][0] + xyxy[0][2]) / 2)
df['center_y'] = df['object_xyxy'].apply(lambda xyxy: (xyxy[0][1] + xyxy[0][3]) / 2)

In [ ]:
resize_factor = 160

# Create a heatmap based on the people's locations (center_x, center_y)
# let's calculate a scaled version of the image (coarse)
shape_image = np.array([w, h]) // resize_factor  # video width and height
plt.figure(figsize=(10, 8))
heatmap_data, xedges, yedges = np.histogram2d(df['center_x'], df['center_y'] , bins=tuple(shape_image), range=[[0, w], [0, h]])
# print(heatmap_data)

sns.heatmap(heatmap_data.T, cmap='coolwarm', square=True, cbar=True)
plt.title("Heatmap of People's Locations Based on Centers")
plt.xlabel("X-axis (center)")
plt.ylabel("Y-axis (center)")

# Display the heatmap
plt.show()

### People Trajectory Analysis

In [ ]:
# normalize values
df_centers = df[["center_x","center_y", "frame_id", "object_id"]].copy()
df_centers["center_x"] = df_centers["center_x"]/ df_centers["center_x"].max()
df_centers["center_y"] = df_centers["center_y"]/ df_centers["center_y"].max()

In [ ]:
from sklearn.cluster import KMeans

# print(df_centers.head(10))
# Sample DataFrame (replace with your actual DataFrame)

df_center_cluster = copy.deepcopy(df_centers)
# DBSCAN Clustering
coords = df_center_cluster[['center_x', 'center_y']].values
db = KMeans(n_clusters=3, random_state=0, n_init='auto').fit(coords)
# Adding cluster labels to the DataFrame
df_center_cluster['cluster'] = db.labels_
# 
# Plotting the clusters
plt.figure(figsize=(10, 6))
plt.scatter(df_center_cluster['center_x'], df_center_cluster['center_y'], c=df_center_cluster['cluster'], cmap='viridis', s=50, alpha=0.7)
plt.xlabel('Center X')
plt.ylabel('Center Y')
plt.title('KMeans Clustering of User Trajectories')
plt.colorbar(label='Cluster')
plt.show()

In [ ]:
# from tslearn.generators import random_walks

# X = random_walks(n_ts=50, sz=32, d=2)
# print(X.shape)

### Time clustering

In [ ]:
grouped_df = df_centers.groupby('object_id').agg({
    'center_x': list,
    'center_y': list
}).reset_index()

grouped_df

In [ ]:
# convert to array list 
temporal_dataset = [[[x,y] for x,y in zip(row["center_x"],row["center_y"])] for idx, row in grouped_df.iterrows()]

In [ ]:
## adapt dataframe
from tslearn.clustering import TimeSeriesKMeans
from tslearn.utils import to_time_series_dataset

## Create a numpy array matrix of (n_ids, n_frames, (coordinates_x,coordinates_y))
time_series_data = to_time_series_dataset(temporal_dataset)
time_series_data.shape

In [ ]:
# K-means clustering
n_clusters = 3  # Number of clusters
kmeans = TimeSeriesKMeans(n_clusters=n_clusters, metric="softdtw", verbose=True)
clusters = kmeans.fit_predict(time_series_data)

print(clusters)
# # Add cluster labels to DataFrame
grouped_df['Cluster'] = clusters

# # Plot the results
plt.figure(figsize=(10, 6))
colors = ['r', 'g', 'b']

counter = {x:0 for x in range(n_clusters)}

for cluster_idx in range(n_clusters):
    clustered_data = grouped_df[grouped_df['Cluster'] == cluster_idx]
    for list_x,list_y in zip(clustered_data["center_x"],clustered_data["center_y"]):
        for x, y in zip(list_x,list_y):
            if counter[cluster_idx] > 0:
                plt.scatter(x, y, color=colors[cluster_idx])
            else:
                plt.scatter(x, y, color=colors[cluster_idx], label=f'Cluster {cluster_idx}')
                counter[cluster_idx] +=1
                
    # print(clustered_data)

plt.xlabel('center_x')
plt.ylabel('center_y')
plt.title('Temporal K-means Clustering on People Center Coordinates')
plt.legend()
plt.show()